In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import numpy as np
from gymnasium import spaces
from tqdm import tqdm

In [2]:
class LatentMDPModel(BaseFeaturesExtractor):
    """
    Custom feature extractor for handling mixed observation spaces (Discrete, MultiDiscrete, Box, Tuple).
    Supports an optional adaptation bottleneck for transfer learning.
    """

    def __init__(
        self,
        observation_space: spaces.Dict,
        input_action_dim: int,
        output_action_dim: int,
        features_dim: int = 64,
        embedding_size: int = 32,
        hidden_dims: list = [128],
        activation_fn: nn.Module = nn.ReLU(),
        dropout: float = 0.1,
        use_residual: bool = True,
        residual_weight_init: float = 1.0,
        load_path: str = None,
    ):
        """
        Initialize the feature extractor.

        Args:
            observation_space (spaces.Dict): The observation space (a dictionary of subspaces).
            features_dim (int): The dimension of the output feature vector.
            embedding_size (int): The size of embeddings for discrete and multi-discrete spaces.
            hidden_dim (int): The default hidden dimension for MLPs.
            hidden_dims (dict): Custom hidden dimensions per key in the observation space.
            activation_fn (nn.Module): The activation function to use.
            adaptation_bottleneck (bool): Whether to use a bottleneck layer for adaptation.
            dropout (float): Dropout probability for regularization.
        """
        super(LatentMDPModel, self).__init__(observation_space, features_dim)

        self.extractors = nn.ModuleDict()
        self.embeddings = nn.ModuleDict()  # Separate embedding layers for MultiDiscrete
        total_embedding_dim = 0

        for key, subspace in observation_space.spaces.items():
            if isinstance(subspace, spaces.MultiDiscrete):
                # Create embeddings separately for each dimension in MultiDiscrete
                embedding_dims = [
                    min(embedding_size, (n // 2) + 1) for n in subspace.nvec
                ]
                self.embeddings[key] = nn.ModuleList(
                    [
                        nn.Embedding(n, dim)
                        for n, dim in zip(subspace.nvec, embedding_dims)
                    ]
                )

                # MLP for processing concatenated embeddings
                input_dim = sum(embedding_dims)

            elif isinstance(subspace, spaces.Box):
                # MLP for Box (continuous) spaces
                input_dim = np.prod(subspace.shape)
            else:
                raise ValueError(
                    f"Unsupported observation space type: {type(subspace)}"
                )
            self.extractors[key] = self.create_mlp(
                input_dim=input_dim,
                hidden_dims=hidden_dims,
                output_dim=features_dim,
                activation_fn=activation_fn,
                norm=True,
                dropout=dropout,
            )
            total_embedding_dim += features_dim

        self.use_residual = use_residual
        if self.use_residual:
            self.residual_layer = nn.Linear(total_embedding_dim, features_dim)
            # Learnable residual weight
            self.residual_weight = nn.Parameter(torch.tensor(residual_weight_init))

        self.fc = nn.Linear(total_embedding_dim, features_dim)

        # State + Next State -> Action
        self.inverse_transition = self.create_mlp(
            input_dim=2 * features_dim,
            hidden_dims=hidden_dims,
            output_dim=output_action_dim,
            activation_fn=activation_fn,
            norm=True,
            dropout=dropout,
        )

        # State + Action -> Next State
        self.transition = self.create_mlp(
            input_dim=features_dim + input_action_dim,
            hidden_dims=hidden_dims,
            output_dim=features_dim,
            activation_fn=activation_fn,
            norm=True,
            dropout=dropout,
        )

        # State + Action -> Reward
        self.reward_prediction = self.create_mlp(
            input_dim=features_dim + input_action_dim,
            hidden_dims=hidden_dims,
            output_dim=1,
            activation_fn=activation_fn,
            norm=True,
            dropout=dropout,
        )

        # Initialize weights
        if load_path is None:
            self.apply(self.init_weights)
        else:
            # print(f"Loading pretrained feature extractor from {load_path}")
            state_dict = torch.load(load_path, weights_only=True)
            filtered_state_dict = {
                k: v for k, v in state_dict.items() if k in self.state_dict()
            }
            self.load_state_dict(filtered_state_dict, strict=False)

    @staticmethod
    def init_weights(m):
        """Initialize weights using Xavier initialization and zero biases."""
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        if isinstance(m, nn.Embedding):
            nn.init.xavier_uniform_(m.weight)

    def create_mlp(
        self, input_dim, hidden_dims, output_dim, activation_fn, norm=True, dropout=0.0
    ):
        """
        Dynamically creates an MLP with optional LayerNorm and Dropout.

        Args:
            input_dim (int): Input feature size.
            hidden_dims (list): List of hidden layer sizes.
            output_dim (int): Output feature size.
            activation_fn (nn.Module): Activation function.
            norm (bool): Whether to apply LayerNorm after each linear layer.
            dropout (float): Dropout probability.

        Returns:
            nn.Sequential: The constructed MLP.
        """
        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(activation_fn)
            if norm:
                layers.append(nn.LayerNorm(hidden_dim))
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        # Final layer (without activation)
        layers.append(nn.Linear(prev_dim, output_dim))

        return nn.Sequential(*layers)

    def to_latent(self, device, observations):
        """
        Forward pass through the feature extractor.

        Args:
            observations (Dict[str, torch.Tensor]): A dictionary of observations.

        Returns:
            torch.Tensor: The extracted features.
        """
        observations = {
            key: obs.to(device) for key, obs in observations.items()
        }  # Move all observations to the device

        # Ensure batch dimension exists
        for key, obs_val in observations.items():
            if len(obs_val.shape) == 1:  # No batch dimension
                observations[key] = obs_val.unsqueeze(0)  # Add batch dimension

        embedded_features = []

        for key, extractor in self.extractors.items():
            obs_val = observations[key]

            if key in self.embeddings:  # MultiDiscrete case
                # Apply embeddings to each dimension and concatenate
                feature_list = [
                    emb(obs_val[:, i].long())
                    for i, emb in enumerate(self.embeddings[key])
                ]
                concatenated_embeddings = torch.cat(feature_list, dim=-1)
                transformed = extractor(concatenated_embeddings)  # Apply MLP
                embedded_features.append(transformed)
            else:  # Box case
                embedded_features.append(
                    extractor(obs_val.view(obs_val.shape[0], -1).float())
                )

        # Concatenate all features
        concatenated_features = torch.cat(embedded_features, dim=-1)
        final_features = self.fc(torch.relu(concatenated_features))

        if self.use_residual:
            # Apply residual connection if enabled
            residual_output = self.residual_layer(torch.relu(concatenated_features))
            # Weight the residual before adding
            final_features += self.residual_weight * residual_output

        # Final projection
        return final_features

    def forward(self, observation, action, next_observation):
        """
        Forward pass through the feature extractor.
        Args:
            observation (dict): Current state observation.
            action (torch.Tensor): Action taken.
            next_observation (dict): Next state observation.
        Returns:
            Tuple of extracted features, predicted action, predicted next state, and predicted reward.
        """
        device = next(self.parameters()).device  # Get model device

        # Move inputs to the same device
        action = action.to(device)
        latent_obs = self.to_latent(device, observation)
        latent_next_obs = self.to_latent(device, next_observation)

        action_pred = self.inverse_transition(
            torch.cat([latent_obs, latent_next_obs], dim=-1).to(device)
        )
        latent_next_obs_pred = self.transition(
            torch.cat([latent_obs, action], dim=-1).to(device)
        )
        reward_pred = self.reward_prediction(
            torch.cat([latent_obs, action], dim=-1).to(device)
        )

        return (
            latent_obs,
            action_pred,
            latent_next_obs,
            latent_next_obs_pred,
            reward_pred,
        )


In [3]:
class DatasetReader(Dataset):
    def __init__(self, file, max_hosts=40, max_jobs=50):
        """
        Args:
            pickle_file (str): Path to the saved .pt file containing the dictionary.
            max_hosts (int): The maximum number of hosts in the observation.
            max_jobs (int): The maximum number of jobs in the observation.
            padding_length (int): Padding length for sequences, default is 3 * 32.
        """
        self.max_hosts = max_hosts
        self.max_jobs = max_jobs
        self.data = self._load_pickle(file)

    def _load_pickle(self, pickle_file):
        """Load data from the pickle file."""
        data = torch.load(pickle_file)  # Load the dataset
        print(f"Loaded dataset with {len(data)} entries")

        # Separate out the dictionary elements
        observations = data["observations"]
        actions = data["actions"]
        rewards = data["rewards"]
        jobs_placed_rewards = data["jobs_placed_rewards"]
        quality_rewards = data["quality_rewards"]
        deadline_violation_rewards = data["deadline_violation_rewards"]
        new_observations = data["new_observations"]

        # Combine all data into one list of tuples (this can be adjusted to your preference)
        combined_data = [
            (obs, action, reward, jobs_reward, quality_reward, deadline_reward, new_obs)
            for obs, action, reward, jobs_reward, quality_reward, deadline_reward, new_obs in zip(
                observations,
                actions,
                rewards,
                jobs_placed_rewards,
                quality_rewards,
                deadline_violation_rewards,
                new_observations,
            )
        ]

        return combined_data

    def __len__(self):
        """Return the length of the dataset (number of data points)."""
        return len(self.data)

    def __getitem__(self, idx):
        """Retrieve a single data point from the dataset."""
        obs, action, reward, jobs_reward, quality_reward, deadline_reward, new_obs = (
            self.data[idx]
        )

        # Apply dynamic padding based on observation keys
        obs = self._apply_dynamic_padding(obs)
        new_obs = self._apply_dynamic_padding(new_obs)

        return {
            "observations": obs,
            "actions": action,
            "rewards": reward,
            "jobs_placed_rewards": jobs_reward,
            "quality_rewards": quality_reward,
            "deadline_violation_rewards": deadline_reward,
            "new_observations": new_obs,
        }

    def _apply_dynamic_padding(self, observation):
        """
        Apply padding to the observation based on the specific keys.

        Pads:
            - dc_ids, dc_types, free_host_pes (for hosts)
            - job_cores, job_location, delay_sensitivity, deadline (for jobs)
        """
        for key, value in observation.items():
            if "dc_" in key or "host" in key:  # These are related to hosts
                num_hosts = len(value)
                pad_size = self.max_hosts - num_hosts
                if pad_size > 0:
                    observation[key] = np.pad(value, (0, pad_size), constant_values=0)
            elif "job_" in key:  # These are related to jobs
                num_jobs = len(value)
                pad_size = self.max_jobs - num_jobs
                if pad_size > 0:
                    observation[key] = np.pad(value, (0, pad_size), constant_values=0)

        return observation

In [7]:
def set_seed_for_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [8]:
def convert_to_tensor(batch):
    """
    Recursively convert all elements of a batch (which could be nested dictionaries) to tensors.

    Args:
    - batch: The batch (could be a dictionary or nested dictionary).

    Returns:
    - The batch with all elements converted to tensors.
    """
    if isinstance(batch, dict):
        # If the batch is a dictionary, recursively convert all values
        return {k: convert_to_tensor(v) for k, v in batch.items()}
    elif isinstance(batch, (list, tuple)):
        # If the batch is a list or tuple, recursively convert all elements
        return [convert_to_tensor(v) for v in batch]
    else:
        # Try to convert the element to a tensor if it's not already a tensor
        tensor = torch.tensor(batch) if not isinstance(batch, torch.Tensor) else batch
        if tensor.dim() == 1:
            tensor = tensor.view(-1, 1)
        return tensor


def move_to_device(batch, device):
    """
    Recursively move elements of a batch (which could contain nested dictionaries) to the specified device (CPU or CUDA),
    after converting them to tensors.

    Args:
    - batch: The batch (could be a dictionary or nested dictionary).
    - device: The target device (e.g., 'cuda' or 'cpu').

    Returns:
    - The batch with all its elements (and nested elements) converted to tensors and moved to the device.
    """
    batch = convert_to_tensor(batch)  # Convert everything to tensors first
    if isinstance(batch, dict):
        # If the batch is a dictionary, recursively move all values
        return {k: move_to_device(v, device) for k, v in batch.items()}
    elif isinstance(batch, (list, tuple)):
        # If the batch is a list or tuple, recursively move all elements
        return [move_to_device(v, device) for v in batch]
    else:
        # If the batch is not a dictionary or list/tuple, assume it's a tensor and move it to the device
        return batch.to(device) if hasattr(batch, "to") else batch


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

seed = 1234
set_seed_for_everything(seed=seed)

g = torch.Generator()
g.manual_seed(seed)

# Load dataset
details_file = "data/all_details.pt"  # Replace with your CSV file path
dataset = DatasetReader(details_file)  # Renamed from TreeDataset
print(f"Dataset size: {len(dataset)}")
subset_fraction = 0.4
subset_size = int(subset_fraction * len(dataset))
dataset = torch.utils.data.Subset(dataset, list(range(subset_size)))
print(f"Subset size: {len(dataset)}")
train_size = int(0.8 * len(dataset))
print(f"Train size: {train_size}")
val_size = len(dataset) - train_size
print(f"Validation size: {val_size}")

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    worker_init_fn=lambda _: seed,
    generator=g,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    worker_init_fn=lambda _: seed,
    generator=g,
)


Using device: cuda


/tmp/ipykernel_1614908/3947284508.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(pickle_file)  # Load the dataset


Loaded dataset with 7 entries
Dataset size: 600041
Subset size: 240016
Train size: 192012
Validation size: 48004


In [15]:
# Loss functions
action_criterion = nn.CrossEntropyLoss()
mse_criterion = nn.MSELoss()


def compute_loss(batch, model, action_max_options, device):
    """Compute the total loss and RMSE for a batch."""
    batch = convert_to_tensor(batch)
    batch = move_to_device(batch, device)

    (
        latent_obs,
        action_pred,
        latent_next_obs,
        latent_next_obs_pred,
        reward_pred,
    ) = model(batch["observations"], batch["actions"], batch["new_observations"])

    actions = batch["actions"]
    rewards = batch["rewards"]

    # Compute individual losses
    action_loss = action_criterion(
        action_pred.view(-1, action_max_options), actions.view(-1)
    )
    reward_loss = mse_criterion(reward_pred, rewards)
    next_obs_loss = mse_criterion(latent_next_obs_pred, latent_next_obs)

    action_coef = 1.0
    reward_coef = 0.0
    next_obs_coef = 0.0

    # Aggregate total loss
    total_loss = (
        action_coef * action_loss
        + reward_coef * reward_loss
        + next_obs_coef * next_obs_loss
    )

    # Compute RMSE (Root Mean Squared Error)
    rmse = torch.sqrt(total_loss)

    return total_loss, rmse


def run_epoch(
    model, data_loader, optimizer, action_max_options, device, is_training=True
):
    """Runs a training or validation epoch and returns loss and RMSE."""
    mode = "Training" if is_training else "Validation"
    model.train() if is_training else model.eval()

    epoch_loss = 0
    epoch_rmse = 0
    total_samples = 0

    with torch.set_grad_enabled(is_training):
        for batch in tqdm(data_loader, desc=f"{mode}...", unit="batch"):
            if is_training:
                optimizer.zero_grad()

            total_loss, rmse = compute_loss(batch, model, action_max_options, device)

            if is_training:
                total_loss.backward()
                optimizer.step()

            batch_size = batch["actions"].size(0)
            total_samples += batch_size
            epoch_loss += total_loss.item() * batch_size  # Weighted sum
            epoch_rmse += rmse.item() * batch_size  # Weighted sum

    # Compute average loss and RMSE
    avg_loss = epoch_loss / total_samples
    avg_rmse = epoch_rmse / total_samples

    print(f"{mode} Loss: {avg_loss:.6f}, {mode} RMSE: {avg_rmse:.6f}")

    return avg_loss, avg_rmse


def train_model(model, optimizer, epochs, train_loader, action_max_options, device):
    """Train the model for a given number of epochs."""
    print(f"Training the Autoencoder for {epochs} epochs")
    for epoch in range(epochs):
        print(f"Epoch [{epoch + 1}/{epochs}]")
        run_epoch(
            model, train_loader, optimizer, action_max_options, device, is_training=True
        )


def validate_model(model, val_loader, action_max_options, device):
    """Validate the model."""
    _, val_rmse = run_epoch(
        model, val_loader, None, action_max_options, device, is_training=False
    )
    return val_rmse


In [16]:
def create_observation_space(
    max_datacenters,
    max_hosts,
    max_datacenter_types,
    max_free_host_pes,
    max_jobs_waiting,
    max_job_delay_sensitivity_levels,
    max_job_cores,
    max_job_deadline,
):
    return spaces.Dict(
        {
            "dc_id": spaces.MultiDiscrete(
                [max_datacenters + 1] * max_hosts,
                dtype=np.int64,
            ),
            "dc_type": spaces.MultiDiscrete(
                [max_datacenter_types + 1] * max_hosts,
                dtype=np.int64,
            ),
            "free_host_pes": spaces.Box(
                low=0,
                high=max_free_host_pes,
                shape=(max_hosts,),
                dtype=np.float32,
            ),
            "job_cores": spaces.Box(
                low=0,
                high=max_job_cores,
                shape=(max_jobs_waiting,),
                dtype=np.float32,
            ),
            "job_location": spaces.MultiDiscrete(
                [max_datacenters + 1] * max_jobs_waiting,
                dtype=np.int64,
            ),
            "job_delay_sensitivity": spaces.MultiDiscrete(
                [max_job_delay_sensitivity_levels + 1] * max_jobs_waiting,
                dtype=np.int64,
            ),
            "job_deadline": spaces.Box(
                low=0,
                high=max_job_deadline,
                shape=(max_jobs_waiting,),
                dtype=np.float32,
            ),
        }
    )

In [17]:
criterion_mse = nn.MSELoss()
criterion_ce = nn.CrossEntropyLoss()  # For action prediction

# Track best configuration
best_val_rmse = float("inf")
best_params = {}
best_model = None

# Define hyperparameters
lr = 0.001
weight_decay = 0
epochs = 5

max_datacenters = 6
max_options = max_datacenters + 1
max_jobs_waiting = 50
input_action_dim = max_jobs_waiting
output_action_dim = max_jobs_waiting * max_options
max_hosts = 40
max_datacenter_types = 3
max_free_host_pes = 64
max_job_delay_sensitivity_levels = 3
max_job_cores = 20
max_job_deadline = 5


observation_space = create_observation_space(
    max_datacenters,
    max_hosts,
    max_datacenter_types,
    max_free_host_pes,
    max_jobs_waiting,
    max_job_delay_sensitivity_levels,
    max_job_cores,
    max_job_deadline,
)

model = LatentMDPModel(
    observation_space=observation_space,
    input_action_dim=input_action_dim,
    output_action_dim=output_action_dim,
).to(device)

# for name, param in model.named_parameters():
#     print(name, param.shape)

optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

# Train the model
train_model(model, optimizer, epochs, train_loader, max_options, device)

# Validate the model
rmse = validate_model(model, val_loader, max_options, device)


Training the Autoencoder for 6 epochs
Epoch [1/6]


Training...: 100%|██████████| 6001/6001 [02:11<00:00, 45.55batch/s]


Training Loss: 1.793415, Training RMSE: 1.339129
Epoch [2/6]


Training...: 100%|██████████| 6001/6001 [02:11<00:00, 45.56batch/s]


Training Loss: 1.777788, Training RMSE: 1.333308
Epoch [3/6]


...

Training Loss: 1.771451, Training RMSE: 1.330929
Epoch [6/6]


Training...: 100%|██████████| 6001/6001 [03:28<00:00, 28.76batch/s]


Training Loss: 1.770629, Training RMSE: 1.330620


Validation...: 100%|██████████| 1501/1501 [00:15<00:00, 98.79batch/s] 

Validation Loss: 1.766249, Validation RMSE: 1.328974


In [18]:
# Save the whole model
torch.save(model.state_dict(), "feature_extractor.pth")

In [ ]:
# ###
# # Testing the best model
# ###

# # Load the best model for testing
# best_model = Autoencoder(
#     input_dim=input_dim,
#     latent_dim=best_params["latent_dim"],
#     dropout=best_params["dropout"],
#     use_batch_norm=best_params["use_batch_norm"],
# )

# best_model.load_state_dict(torch.load("best_autoencoder.pth"))
# best_model.eval()

# test_loss = 0
# test_rmse = 0
# total_samples = 0

# test_loader = DataLoader(
#     test_dataset, batch_size=best_params["batch_size"], shuffle=False
# )

# with torch.no_grad():
#     for batch in tqdm(test_loader, desc="Testing...", unit="batch"):
#         _, reconstructed = best_model(batch)
#         loss = criterion(reconstructed, batch)
#         batch_size = batch.size(0)
#         total_samples += batch_size

#         test_loss += loss.item() * batch_size

#         rmse = torch.sqrt(loss)
#         test_rmse += rmse.item() * batch_size

# test_loss /= total_samples
# test_rmse /= total_samples

# print(f"Test Loss: {test_loss:.6f}, Test RMSE: {test_rmse:.6f}")


In [ ]:
model = LatentMDPModel(
    observation_space=observation_space,
    input_action_dim=input_action_dim,
    output_action_dim=output_action_dim,
).to(device)

In [ ]:
missing, unexpected = model.load_state_dict(
    torch.load("feature_extractor.pth", weights_only=True), strict=False
)
if missing:
    print(f"Missing keys: {missing}")
if unexpected:
    print(f"Unexpected keys: {unexpected}")